# Lab Feature Test

`services/forest_mask/cli.py` の特徴量分析セクションのうち、Lab 変換のみを実行する。

- 入力: `data/input/jpg/sample/sample_tree.jpg`
- 出力: `data/intermediate/exg/lab{l,a,b}_<timestamp>.jpg`

Lab チャンネルの意味:

- L : 0-255 (明度)
- a : 0-255 (128 が中心、 <128 = 緑、 >128 = 赤)
- b : 0-255 (128 が中心、 <128 = 青、 >128 = 黄)

In [ ]:
# プロジェクトルートで実行できるようにカレントを移動
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
    os.chdir(PROJECT_ROOT)

print("cwd:", Path.cwd())

In [ ]:
# モジュールリロードを有効化（実装変更を即反映するため）
%load_ext autoreload
%autoreload 2

In [ ]:
# Lab 計算と保存
from datetime import datetime

from services.common.image import load_rgb, save_gray
from services.forest_mask.color import calc_lab

INPUT_PATH = Path("data/input/jpg/sample/sample_tree.jpg")

rgb = load_rgb(INPUT_PATH)
lab_l, lab_a, lab_b = calc_lab(rgb)

formatted_time = datetime.now().strftime("%Y%m%d%H%M%S")
output_dir = Path("data/intermediate/exg")
output_dir.mkdir(parents=True, exist_ok=True)

lab_l_path = output_dir / f"labl_{formatted_time}.jpg"
lab_a_path = output_dir / f"laba_{formatted_time}.jpg"
lab_b_path = output_dir / f"labb_{formatted_time}.jpg"

save_gray(lab_l, lab_l_path)
save_gray(lab_a, lab_a_path)
save_gray(lab_b, lab_b_path)

print(f"Saved: {lab_l_path}")
print(f"Saved: {lab_a_path}")
print(f"Saved: {lab_b_path}")
for name, ch in [("L", lab_l), ("a", lab_a), ("b", lab_b)]:
    print(f"{name}: shape={ch.shape} dtype={ch.dtype} min={int(ch.min())} max={int(ch.max())}")

In [ ]:
# 入力と L, a, b を並べて表示
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
axes[0].imshow(rgb)
axes[0].set_title("input (RGB)")
axes[0].axis("off")

for ax, ch, name in [
    (axes[1], lab_l, "L"),
    (axes[2], lab_a, "a (green<128<red)"),
    (axes[3], lab_b, "b (blue<128<yellow)"),
]:
    im = ax.imshow(ch, cmap="gray")
    ax.set_title(name)
    ax.axis("off")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()